In [1]:
from pyomo.environ import (
    ConcreteModel,
    value,
    assert_optimal_termination,
    units as pyunits,
)
from idaes.core import MaterialFlowBasis

from watertap.property_models.multicomp_aq_sol_prop_pack import (
    MCASParameterBlock,
)
from pyomo.network import Port
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
from watertap_contrib.reflo.unit_models.zero_order.chemical_softening_zo import (
    ChemicalSofteningZO,
)
from watertap_contrib.reflo.costing import TreatmentCosting

from idaes.core.util.testing import initialization_tester
from idaes.core.solvers import get_solver
from idaes.core.util.model_statistics import (
    degrees_of_freedom,
    number_variables,
    number_total_constraints,
    number_unused_variables,
)
from idaes.core.util.scaling import (
    calculate_scaling_factors,
    unscaled_variables_generator,
    badly_scaled_var_generator,
)
import idaes.logger as idaeslog
from watertap.core.util.model_diagnostics.infeasible import *

# Get default solver for testing
solver = get_solver()

In [2]:
component_list = ["Ca_2+", "Mg_2+","SiO2","Alkalinity_2-"]
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.properties = MCASParameterBlock(
    solute_list=component_list, material_flow_basis=MaterialFlowBasis.mass
)

m.fs.soft = soft = ChemicalSofteningZO(
    property_package=m.fs.properties,
    silica_removal=True,
    softening_procedure_type="excess_lime",
)

ca_in = 0.61 * pyunits.kg / pyunits.m**3  # g/L = kg/m3
mg_in = 0.161 * pyunits.kg / pyunits.m**3  # g/L = kg/m3
sio2_in = 0.13 * pyunits.kg / pyunits.m**3  # g/L = kg/m3
alk_in = 0.0821 * pyunits.kg / pyunits.m**3  # g/L = kg/m3
CO2_in = 0.10844915* pyunits.kg / pyunits.m**3
# q_in = 3785 * pyunits.m**3 / pyunits.day  # m3/d
q_in = 1 * pyunits.L / pyunits.s
rho = 1000 * pyunits.kg / pyunits.m**3

prop_in = soft.properties_in[0]
prop_out = soft.properties_out[0]

flow_mass_phase_water = pyunits.convert(
    q_in * rho, to_units=pyunits.kg / pyunits.s
)
flow_mass_phase_ca = pyunits.convert(
    q_in * ca_in, to_units=pyunits.kg / pyunits.s
)
flow_mass_phase_mg = pyunits.convert(
    q_in * mg_in, to_units=pyunits.kg / pyunits.s
)
flow_mass_phase_si = pyunits.convert(
    q_in * sio2_in, to_units=pyunits.kg / pyunits.s
)
flow_mass_phase_alk = pyunits.convert(
    q_in * alk_in, to_units=pyunits.kg / pyunits.s
)
prop_in.flow_mass_phase_comp["Liq", "H2O"].fix(flow_mass_phase_water)

prop_in.flow_mass_phase_comp["Liq", "Ca_2+"].fix(flow_mass_phase_ca)
prop_in.flow_mass_phase_comp["Liq", "Mg_2+"].fix(flow_mass_phase_mg)
prop_in.flow_mass_phase_comp["Liq", "SiO2"].fix(flow_mass_phase_si)
prop_in.flow_mass_phase_comp["Liq", "Alkalinity_2-"].fix(
    flow_mass_phase_alk
)

prop_in.temperature.fix()
prop_in.pressure.fix()

#Ca to CaCO3 conv = 2.5
ca_eff_target = ca_in()*10
#Mg to CaCO3 conv = 4.12
mg_eff_target = mg_in()*0.1

soft.ca_eff_target.fix(1.312938301706484)
soft.mg_eff_target.fix(4.2551166976399655)

soft.no_of_mixer.fix(1)
soft.no_of_floc.fix(2)
soft.retention_time_mixer.fix(0.4)
soft.retention_time_floc.fix(25)
soft.retention_time_sed.fix(130)
soft.retention_time_recarb.fix(20)
soft.frac_vol_recovery.fix()
soft.removal_efficiency.fix()
soft.CO2_CaCO3.fix(CO2_in)
soft.vel_gradient_mix.fix(300)
soft.vel_gradient_floc.fix(50)
# soft.excess_CaO.fix(0)
soft.CO2_second_basin.fix(0)
soft.Na2CO3_dosing.fix(0)
# soft.MgCl2_dosing.fix()

m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp",
    value(1 / flow_mass_phase_water),
    index=("Liq", "H2O"),
)
m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp",
    value(1 / flow_mass_phase_ca),
    index=("Liq", "Ca_2+"),
)
m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp",
    value(1 / flow_mass_phase_mg),
    index=("Liq", "Mg_2+"),
)
m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp", value(1 / flow_mass_phase_si), index=("Liq", "SiO2")
)
m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp",
    value(1 / flow_mass_phase_alk),
    index=("Liq", "Alkalinity_2-"),
)

soft.properties_waste[0].conc_mass_phase_comp["Liq", "Mg_2+"]
soft.properties_out[0].conc_mass_phase_comp["Liq", "Ca_2+"]
print(f"DOF = {degrees_of_freedom(m)}")

DOF = 0


In [3]:
soft.initialize()
print(f"DOF = {degrees_of_freedom(m)}")

2024-04-03 07:27:14 [INFO] idaes.init.fs.soft: Initialization Step 1a Complete.
2024-04-03 07:27:14 [INFO] idaes.init.fs.soft: Initialization Step 1b Complete.
2024-04-03 07:27:14 [INFO] idaes.init.fs.soft: Initialization Step 1c Complete.
2024-04-03 07:27:14 [INFO] idaes.init.fs.soft: Initialization Step 2 optimal - Optimal Solution Found.
2024-04-03 07:27:14 [INFO] idaes.init.fs.soft: Initialization Complete: optimal - Optimal Solution Found
DOF = 0


In [4]:
print(soft.ca_eff_target())
print(soft.mg_eff_target())

1.312938301706484
4.2551166976399655


In [5]:
print_infeasible_constraints(m)

In [6]:
print(soft.frac_vol_recovery())
print(soft.properties_in[0].flow_vol_phase["Liq"]())
print(soft.properties_out[0].flow_vol_phase["Liq"]())
print(soft.properties_waste[0].flow_vol_phase["Liq"]())

0.99
0.0010009831000000005
0.0009909732690000005
1.0009830999999946e-05


In [7]:
results = solver.solve(m)

In [8]:
print('Ca Flow mass in kg/s:')
print('In:',soft.properties_in[0].flow_mass_phase_comp["Liq", "Ca_2+"]())
print('Out:',soft.properties_out[0].flow_mass_phase_comp["Liq", "Ca_2+"]())
print('Waste:',soft.properties_waste[0].flow_mass_phase_comp["Liq", "Ca_2+"]())

print('\nCa Conc mass in kg/s:')
print('In:',soft.properties_in[0].conc_mass_phase_comp["Liq", "Ca_2+"]())
print('Out:',soft.properties_out[0].conc_mass_phase_comp["Liq", "Ca_2+"]())
print('Waste:',soft.properties_waste[0].conc_mass_phase_comp["Liq", "Ca_2+"]())
print('excess cao:',soft.excess_CaO())


Ca Flow mass in kg/s:
In: 0.0006100000000000001
Out: 0.0005204347043349532
Waste: 0.00021766166061895165

Ca Conc mass in kg/s:
In: 0.6094008979771985
Out: 0.5251753206825936
Waste: 21.744788760065166
excess cao: 0.1279705570992205


In [9]:
print('Mg Flow mass in kg/s:')
print('In:',soft.properties_in[0].flow_mass_phase_comp["Liq", "Mg_2+"]())
print('Out:',soft.properties_out[0].flow_mass_phase_comp["Liq", "Mg_2+"]())
print('Waste:',soft.properties_waste[0].flow_mass_phase_comp["Liq", "Mg_2+"]())

print('\nMg Conc mass in kg/s:')
print('In:',soft.properties_in[0].conc_mass_phase_comp["Liq", "Mg_2+"]())
print('Out:',soft.properties_out[0].conc_mass_phase_comp["Liq", "Mg_2+"]())
print('Waste:',soft.properties_waste[0].conc_mass_phase_comp["Liq", "Mg_2+"]())
print('MgCl2:',soft.MgCl2_dosing())

Mg Flow mass in kg/s:
In: 0.00016100000000000004
Out: 0.001023472549474942
Waste: 0.0003335274505250585

Mg Conc mass in kg/s:
In: 0.16084187635135896
Out: 1.0327953149611566
Waste: 33.31998817213386
MgCl2: 1.194825367181524


In [10]:
print(soft.ca_eff_target())
print(soft.mg_eff_target())

1.312938301706484
4.2551166976399655


In [11]:
print(f"DOF = {degrees_of_freedom(m)}")

DOF = 0
